# Fine-tune BERT for IPC statute identification (ILSI)

Multi-label classification of Indian case facts into 100 IPC sections. Same label order, metrics and threshold tuning as the Mac baselines (`metrics.py` ships inside the upload zip).

## How to run
1. Upload **`colab_upload.zip`** (from the project root on the Mac) to Google Drive at **`MyDrive/ilsi/colab_upload.zip`**.
2. *Runtime → Change runtime type →* **GPU** (T4 works; L4/A100 are faster).
3. **First run a smoke test:** set `SMOKE_TEST = True` in the config cell and *Runtime → Run all* (≈5 min, most of it is the one-off setup). Check that the last cell prints a metrics table.
4. Set `SMOKE_TEST = False`, choose `MODEL_NAME` and `TRUNCATION`, then *Run all*.
5. **If Colab disconnects**, reconnect and *Run all* again: tokenized data is cached on Drive and training resumes from the latest checkpoint.
6. Download **`MyDrive/ilsi/runs/<run_name>/<run_name>_results.zip`** (small). For the best model also download the **`model/`** folder (≈420 MB) for the demo app / BNS experiment.

Run name = `{model_short}_{truncation}_{maxlen}`, e.g. `inlegalbert_head_512`.

## Expected runtime per full run (rough estimates, not measured)
| GPU | 4 epochs, 512 tokens, 42.8k docs |
|---|---|
| T4 (free tier) | ~1.5–2.5 h (+ ~5 min first-time tokenization) |
| L4 | ~45–75 min |
| A100 | ~20–40 min |

Drive space: up to 2 checkpoints × ~1.3 GB (model + optimizer state) plus the final model (~0.42 GB) per run. Delete `MyDrive/ilsi/checkpoints/<run_name>` after you've downloaded results.

## 1. Config

In [ ]:
import os

MODEL_NAME = "law-ai/InLegalBERT"   # also run: "bert-base-uncased"
TRUNCATION = "head"                 # "head" = first 510 tokens; "head_tail" = first 128 + last 382
MAX_LEN = 512
HEAD_TOKENS = 128                   # head_tail only
EPOCHS = 4
LR = 2e-5
BATCH = 16
GRAD_ACCUM = 2                      # effective batch = 32
WARMUP = 0.1                        # fraction of total steps
WEIGHT_DECAY = 0.01
USE_POS_WEIGHT = True
POS_WEIGHT_CLIP = 30.0
BEST_METRIC = "micro_f1"            # checkpoint selection on dev: "micro_f1" or "macro_f1"
SMOKE_TEST = False                  # True: 500 train / 200 dev / 200 test rows, MAX_LEN=128, 1 epoch
DRIVE_DIR = "/content/drive/MyDrive/ilsi"
SEED = 42

# Optional environment overrides, used only for local validation on the Mac. Leave as is in Colab.
MODEL_NAME = os.environ.get("ILSI_MODEL_NAME", MODEL_NAME)
TRUNCATION = os.environ.get("ILSI_TRUNCATION", TRUNCATION)
SMOKE_TEST = os.environ.get("ILSI_SMOKE_TEST", "1" if SMOKE_TEST else "0") == "1"
DRIVE_DIR = os.environ.get("ILSI_DRIVE_DIR", DRIVE_DIR)

assert TRUNCATION in {"head", "head_tail"}
assert BEST_METRIC in {"micro_f1", "macro_f1"}
N_TRAIN = N_DEV = N_TEST = None
if SMOKE_TEST:
    MAX_LEN, EPOCHS = 128, 1
    N_TRAIN, N_DEV, N_TEST = 500, 200, 200
if MAX_LEN != 512:                  # keep the 1:3 head:tail split of 128:382 for other lengths
    HEAD_TOKENS = (MAX_LEN - 2) // 4
MODEL_SHORT = MODEL_NAME.split("/")[-1].lower()
RUN_NAME = f"{MODEL_SHORT}_{TRUNCATION}_{MAX_LEN}" + ("_smoke" if SMOKE_TEST else "")
print("RUN_NAME:", RUN_NAME, "| SMOKE_TEST:", SMOKE_TEST)

## 2. Setup: Drive, data, packages, GPU

In [ ]:
import sys, subprocess, zipfile, importlib.metadata as md
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

PINNED = {"transformers": "5.20.2", "datasets": "5.1.0", "accelerate": "1.15.0"}
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    def _ver(p):
        try:
            return md.version(p)
        except md.PackageNotFoundError:
            return None
    need = [f"{p}=={v}" for p, v in PINNED.items() if _ver(p) != v]
    if need:
        print("installing", need)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=True)
        print("If the next cell fails with an import error: Runtime -> Restart session, then Run all.")

DRIVE = Path(DRIVE_DIR)
DRIVE.mkdir(parents=True, exist_ok=True)
ZIP_PATH = DRIVE / "colab_upload.zip"
assert ZIP_PATH.exists(), f"Upload colab_upload.zip to {ZIP_PATH} first"
LOCAL = Path("/content/ilsi_data") if IN_COLAB else DRIVE / "_unzipped"
DATA_DIR = LOCAL / "colab_upload"
if not (DATA_DIR / "label_list.json").exists():
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(LOCAL)
sys.path.insert(0, str(DATA_DIR))
print("data dir:", DATA_DIR, sorted(p.name for p in DATA_DIR.iterdir()))

import torch
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else (
    "Apple MPS" if torch.backends.mps.is_available() else "CPU")
print("device:", GPU_NAME, "| torch", torch.__version__)
if IN_COLAB and not torch.cuda.is_available():
    raise RuntimeError("No GPU. Runtime -> Change runtime type -> GPU")

## 3. Load data

In [ ]:
import json, math, shutil, time
import numpy as np
import pandas as pd
import transformers, datasets
from transformers import (AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
                          Trainer, TrainingArguments, set_seed)
from transformers.trainer_utils import get_last_checkpoint
from datasets import Dataset, DatasetDict, load_from_disk
import metrics as M

set_seed(SEED)
print("transformers", transformers.__version__, "| datasets", datasets.__version__)
label_list = json.load(open(DATA_DIR / "label_list.json"))
label_buckets = json.load(open(DATA_DIR / "label_buckets.json"))
L = len(label_list)

def load_split(split, n):
    df = pd.read_parquet(DATA_DIR / f"{split}.parquet")
    if n is not None:
        df = df.sample(n=min(n, len(df)), random_state=SEED).sort_index()
    df = df.reset_index(drop=True)
    df["labels"] = df["labels"].map(list)
    return df

dfs = {"train": load_split("train", N_TRAIN), "dev": load_split("dev", N_DEV), "test": load_split("test", N_TEST)}
Y = {s: M.binarize(d["labels"], label_list) for s, d in dfs.items()}
print({s: len(d) for s, d in dfs.items()}, "labels:", L)

pos = Y["train"].sum(0)
pos_weight = np.clip((len(Y["train"]) - pos) / np.maximum(pos, 1), 1.0, POS_WEIGHT_CLIP).astype(np.float32)
print(f"pos_weight: min {pos_weight.min():.1f}, median {np.median(pos_weight):.1f}, max {pos_weight.max():.1f}, "
      f"clipped at {POS_WEIGHT_CLIP}: {(pos_weight >= POS_WEIGHT_CLIP).sum()} labels")

## 4. Tokenize (cached on Drive)

`head`: the tokenizer keeps the first MAX_LEN-2 tokens. `head_tail`: tokenize the whole document, then keep the first HEAD_TOKENS and the last MAX_LEN-2-HEAD_TOKENS tokens (128 + 382 at 512); documents that fit are left unchanged.

In [ ]:
try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
except ValueError:
    # repos that ship only vocab.txt with no tokenizer_config.json (e.g. prajjwal1/bert-tiny)
    from transformers import BertTokenizer
    tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
TOK_CACHE = DRIVE / "tok_cache" / RUN_NAME
N_SPECIAL = tokenizer.num_special_tokens_to_add(pair=False)
BUDGET = MAX_LEN - N_SPECIAL
TAIL_TOKENS = BUDGET - HEAD_TOKENS
assert 0 < HEAD_TOKENS < BUDGET
USE_TT = "token_type_ids" in tokenizer.model_input_names
CLS_ID, SEP_ID = tokenizer.cls_token_id, tokenizer.sep_token_id
if TRUNCATION == "head_tail":
    assert N_SPECIAL == 2 and CLS_ID is not None and SEP_ID is not None, \
        "head_tail assumes BERT-style [CLS] tokens [SEP] inputs"

def clip_chars(t, n=200_000):
    # guard against a few pathological >400k-char documents; both ends are kept
    return t if len(t) <= 2 * n else t[:n] + " " + t[-n:]

def encode(batch):
    texts = [clip_chars(t) for t in batch["text"]]
    if TRUNCATION == "head":
        enc = tokenizer(texts, truncation=True, max_length=MAX_LEN)
        out = {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"]}
        if USE_TT:
            out["token_type_ids"] = enc["token_type_ids"]
    else:
        raw = tokenizer(texts, add_special_tokens=False, truncation=False)["input_ids"]
        ids = []
        for r in raw:
            if len(r) > BUDGET:
                r = r[:HEAD_TOKENS] + r[-TAIL_TOKENS:]
            ids.append([CLS_ID] + r + [SEP_ID])
        out = {"input_ids": ids, "attention_mask": [[1] * len(x) for x in ids]}
        if USE_TT:
            out["token_type_ids"] = [[0] * len(x) for x in ids]
    out["n_tokens"] = [len(x) for x in out["input_ids"]]
    return out

if (TOK_CACHE / "dataset_dict.json").exists():
    tok = load_from_disk(str(TOK_CACHE))
    print("loaded tokenized cache from", TOK_CACHE)
else:
    transformers.logging.set_verbosity_error()   # silence "sequence longer than max" warnings
    raw_ds = DatasetDict({s: Dataset.from_dict({"text": d["text"].tolist(),
                                                "labels": Y[s].astype(np.float32).tolist()})
                          for s, d in dfs.items()})
    tok = raw_ds.map(encode, batched=True, batch_size=256, remove_columns=["text"], desc="tokenize")
    transformers.logging.set_verbosity_warning()
    tmp = TOK_CACHE.with_name(TOK_CACHE.name + "_tmp")
    shutil.rmtree(tmp, ignore_errors=True)
    tok.save_to_disk(str(tmp))
    shutil.rmtree(TOK_CACHE, ignore_errors=True)
    tmp.rename(TOK_CACHE)                          # atomic-ish: a half-written cache is never used
    print("saved tokenized cache to", TOK_CACHE)

for s in tok:
    n = np.array(tok[s]["n_tokens"])
    assert len(tok[s]) == len(dfs[s]), "tokenized/data row mismatch"
    print(f"{s}: {len(n)} docs, tokens mean {n.mean():.0f}, at max length: {100 * (n >= MAX_LEN).mean():.1f}%")
tok = tok.remove_columns("n_tokens")
ex = tok["train"][0]
print("example:", tokenizer.decode(ex["input_ids"][:40]), "...")

## 5. Model and trainer (BCE with pos_weight)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=L, problem_type="multi_label_classification",
    id2label=dict(enumerate(label_list)), label2id={l: i for i, l in enumerate(label_list)})

class WeightedBCETrainer(Trainer):
    def __init__(self, *args, pos_weight=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.pos_weight = pos_weight
        # Our loss is a per-batch mean, so let the Trainer divide by grad-accumulation steps.
        self.model_accepts_loss_kwargs = False

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits.float()
        pw = self.pos_weight.to(logits.device) if self.pos_weight is not None else None
        loss = torch.nn.functional.binary_cross_entropy_with_logits(logits, labels.float(), pos_weight=pw)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits = eval_pred.predictions[0] if isinstance(eval_pred.predictions, tuple) else eval_pred.predictions
    prob = 1 / (1 + np.exp(-logits.astype(np.float32)))
    m = M.evaluate(eval_pred.label_ids, prob, 0.5)
    return {"micro_f1": m["micro_f1"], "macro_f1": m["macro_f1"]}

CKPT_DIR = DRIVE / "checkpoints" / RUN_NAME
args = TrainingArguments(
    output_dir=str(CKPT_DIR),
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    per_device_train_batch_size=BATCH,
    per_device_eval_batch_size=BATCH * 2,
    gradient_accumulation_steps=GRAD_ACCUM,
    warmup_steps=WARMUP,                 # float in [0,1) = ratio of total steps
    weight_decay=WEIGHT_DECAY,
    lr_scheduler_type="linear",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model=BEST_METRIC,
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    logging_strategy="steps",
    logging_steps=10 if SMOKE_TEST else 50,
    logging_first_step=True,
    report_to="none",
    seed=SEED,
    data_seed=SEED,
    dataloader_num_workers=2 if torch.cuda.is_available() else 0,
)
trainer = WeightedBCETrainer(
    model=model, args=args, train_dataset=tok["train"], eval_dataset=tok["dev"],
    processing_class=tokenizer, data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
    pos_weight=torch.tensor(pos_weight) if USE_POS_WEIGHT else None)

## 6. Train (auto-resumes from the latest checkpoint on Drive)

In [ ]:
last_ckpt = get_last_checkpoint(str(CKPT_DIR)) if CKPT_DIR.exists() else None
print("resume from:", last_ckpt)
t0 = time.time()
train_result = trainer.train(resume_from_checkpoint=last_ckpt)
TRAIN_SECONDS = time.time() - t0
print(f"training done in {TRAIN_SECONDS / 60:.1f} min; best checkpoint: {trainer.state.best_model_checkpoint} "
      f"({BEST_METRIC}={trainer.state.best_metric})")

# per-epoch log
rows = {}
for e in trainer.state.log_history:
    if "epoch" not in e:
        continue
    ep = max(1, math.ceil(e["epoch"] - 1e-9))
    r = rows.setdefault(ep, {"epoch": ep, "train_losses": []})
    if "loss" in e:
        r["train_losses"].append(e["loss"])
        r["learning_rate"] = e.get("learning_rate")
    if "eval_micro_f1" in e:
        r.update(eval_loss=e.get("eval_loss"), eval_micro_f1_t05=e["eval_micro_f1"],
                 eval_macro_f1_t05=e["eval_macro_f1"])
train_log = pd.DataFrame([{**{k: v for k, v in r.items() if k != "train_losses"},
                           "train_loss": float(np.mean(r["train_losses"])) if r["train_losses"] else None}
                          for _, r in sorted(rows.items())])
train_log

## 7. Predict dev/test, tune thresholds on dev, evaluate on test

In [ ]:
def predict_probs(ds):
    out = trainer.predict(ds)
    logits = out.predictions[0] if isinstance(out.predictions, tuple) else out.predictions
    return (1 / (1 + np.exp(-logits.astype(np.float32)))).astype(np.float32)

prob_dev, prob_test = predict_probs(tok["dev"]), predict_probs(tok["test"])
assert prob_dev.shape == (len(dfs["dev"]), L) and prob_test.shape == (len(dfs["test"]), L)
metrics, thresholds = M.full_report(Y["dev"], prob_dev, Y["test"], prob_test, label_list, label_buckets)

summary = pd.DataFrame([{"thresholds": k,
                         "dev_micro_f1": v["dev"]["micro_f1"], "dev_macro_f1": v["dev"]["macro_f1"],
                         "test_micro_f1": v["test"]["micro_f1"], "test_macro_f1": v["test"]["macro_f1"],
                         **{f"test_{b}_f1": x for b, x in v["test"]["buckets"].items()}}
                        for k, v in metrics.items()])
print(f"global threshold (tuned on dev): {thresholds['global']}")
print(f"DEV micro-F1 at global threshold = {metrics['global']['dev']['micro_f1']:.4f}  "
      "<- compare runs on this (dev), not on test")
print(summary.round(4).to_string(index=False))

## 8. Save results to Drive and zip them

In [ ]:
RUN_DIR = DRIVE / "runs" / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
config = {
    "run_name": RUN_NAME, "model_name": MODEL_NAME, "truncation": TRUNCATION, "max_len": MAX_LEN,
    "head_tokens": HEAD_TOKENS if TRUNCATION == "head_tail" else None,
    "tail_tokens": TAIL_TOKENS if TRUNCATION == "head_tail" else None,
    "epochs": EPOCHS, "lr": LR, "batch": BATCH, "grad_accum": GRAD_ACCUM, "warmup": WARMUP,
    "weight_decay": WEIGHT_DECAY, "use_pos_weight": USE_POS_WEIGHT, "pos_weight_clip": POS_WEIGHT_CLIP,
    "best_metric": BEST_METRIC, "smoke_test": SMOKE_TEST, "seed": SEED,
    "n_rows": {s: len(d) for s, d in dfs.items()}, "n_labels": L, "label_list": label_list,
    "device": GPU_NAME, "train_minutes": round(TRAIN_SECONDS / 60, 1),
    "best_checkpoint": trainer.state.best_model_checkpoint, "best_dev_metric_t05": trainer.state.best_metric,
    "versions": {"torch": torch.__version__, "transformers": transformers.__version__, "datasets": datasets.__version__},
}
M.save_outputs(RUN_DIR, prob_dev, prob_test, thresholds, metrics, extra={"run_name": RUN_NAME})
train_log.to_csv(RUN_DIR / "train_log.csv", index=False)
(RUN_DIR / "config.json").write_text(json.dumps(config, indent=1))
(RUN_DIR / "split_ids.json").write_text(json.dumps({s: dfs[s]["id"].tolist() for s in ["dev", "test"]}))

trainer.save_model(str(RUN_DIR / "model"))          # best checkpoint (load_best_model_at_end=True)
tokenizer.save_pretrained(str(RUN_DIR / "model"))

zip_path = RUN_DIR / f"{RUN_NAME}_results.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(RUN_DIR.iterdir()):
        if f.is_file() and f != zip_path:
            z.write(f, arcname=f"{RUN_NAME}/{f.name}")
print("saved:", sorted(p.name for p in RUN_DIR.iterdir()))
print(f"\nDOWNLOAD: {zip_path}  ({zip_path.stat().st_size / 1e6:.1f} MB)")
print(f"model weights (optional, best run only): {RUN_DIR / 'model'}")